# Imports

In [23]:
import kagglehub
import pandas as pd
import os

# Dowload file from kaggle

In [ ]:

source_file = kagglehub.dataset_download("patelris/crop-yield-prediction-dataset", path="yield_df.csv")

print("Path to dataset file:", source_file)

Path to dataset files: /home/jiivanmukta/.cache/kagglehub/datasets/patelris/crop-yield-prediction-dataset/versions/7/yield_df.csv


# Splitting the file into N parts

In [24]:
def split_csv_into_n_parts(original_file_name, n_parts=3, base_dir='spark-data', filename='data', has_index=True, verbose=True):
    # Count total data rows (subtract 1 for the header)
    with open(original_file_name, 'rb') as f:
        total_rows = sum(1 for _ in f) - 1

    if total_rows <= 0:
        raise ValueError("The CSV file is empty or only contains a header.")

    # Calculate exact row counts per file
    base_size, remainder = divmod(total_rows, n_parts)

    # Stream the CSV and save each chunk into its node directory
    with pd.read_csv(original_file_name, index_col=0 if has_index else None, iterator=True) as reader:
        for i in range(n_parts):
            chunk_size = base_size + (1 if i < remainder else 0)
            if chunk_size == 0:
                break

            # First part goes to 'master', subsequent parts go to 'worker1', 'worker2', etc.
            node_folder = 'master' if i == 0 else f'worker{i}'
            target_dir = os.path.join(base_dir, node_folder)

            # Create the folder structure if it doesn't exist
            os.makedirs(target_dir, exist_ok=True)

            # Build full file path (e.g., spark-data/master/data_0.csv)
            output_path = os.path.join(target_dir, f'{filename}_{i}.csv')

            chunk = reader.get_chunk(chunk_size)
            chunk.to_csv(output_path, index=False)

            if verbose:
                print(f'Successfully created {output_path} ({len(chunk)} rows)')



In [25]:
# Example usage for 1 master + 2 workers (3 parts total):
split_csv_into_n_parts(source_file, n_parts=3, base_dir='spark-data', filename='data', has_index=True, verbose=True)

Successfully created spark-data/master/data_0.csv (9414 rows)
Successfully created spark-data/worker1/data_1.csv (9414 rows)
Successfully created spark-data/worker2/data_2.csv (9414 rows)
